In [8]:
### Q1. Simple Linear Classifier

import torch
import torch.nn as nn
import torch.optim as optim

input_dim = 1
output_dim = 1
learning_rate = 0.01
num_epochs = 100

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = 2*x + 1

class LinearRegressionModel(nn.Module):
  def __init__(self, input_dim, output_dim):
    super(LinearRegressionModel, self).__init__()
    self.linear = nn.Linear(input_dim, output_dim)

  def forward(self, x):
    output = self.linear(x)
    return output

model = LinearRegressionModel(input_dim, output_dim)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
  optimizer.zero_grad()
  pred = model(x)
  loss = criterion(pred, y)
  loss.backward()
  optimizer.step()

  if epoch % 20 == 0:
    print(f'Epoch {epoch}: loss {loss:.4f}')


Epoch 0: loss 42.1971
Epoch 20: loss 0.0723
Epoch 40: loss 0.0391
Epoch 60: loss 0.0347
Epoch 80: loss 0.0307


In [9]:
### Q2. DataLoader & Preprocessing

from torchvision import datasets, transforms
from torch.utils.data import DataLoader


## 1. 전처리 Transform 구현 - 텐서화 & 정규화 & 데이터 증강
trained_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)), # 정규화
    transforms.RandomResizedCrop(28), # 무작위 크기 조절 및 크롭
    transforms.RandomHorizontalFlip(p=0.5), # 무작위 좌우 반전
    transforms.RandomRotation(degrees=15), # 무작위 각도 회전
])



## 2. MNIST 훈련 & 테스트 데이터세트 호출 & 전처리 적용
train = datasets.MNIST(root='./data', train=True, download=True,
                       transform=trained_transform)
test = datasets.MNIST(root='./data', train=False, download=True,
                      transform=trained_transform)

## 3. 전처리된 datasets를 DataLoader에 투입
train_loader = DataLoader(train, batch_size=64, shuffle=True)
test_loader = DataLoader(test, batch_size=64, shuffle=False)


## 4. train data 첫번째 배치 shape 확인
images, labels = next(iter(train_loader))
print(f"이미지 배치 shape: {images.shape}")
print(f"레이블 배치 shape: {labels.shape}")
print(f"첫 번째 샘플 레이블: {labels[0]}")



이미지 배치 shape: torch.Size([64, 1, 28, 28])
레이블 배치 shape: torch.Size([64])
첫 번째 샘플 레이블: 0


In [10]:
### Q3. Simple MLP Model & evaluation function

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# GPU 장치 설정 추가
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 단순 MLP 모델 정의 - Linear → ReLU → Linear

class SimpleMLP(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(SimpleMLP, self).__init__()
    self.input_dim = input_dim
    self.fc = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim)
    )

  def forward(self, x):
      x = x.view(-1, self.input_dim)
      return self.fc(x)


## 2. 훈련/테스트 데이터 호출 & 전처리/데이터로더 투입
trained_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)), # 정규화
    transforms.RandomResizedCrop(28), # 무작위 크기 조절 및 크롭
    transforms.RandomHorizontalFlip(p=0.5), # 무작위 좌우 반전
    transforms.RandomRotation(degrees=15), # 무작위 각도 회전
])

train = datasets.MNIST(root='./data', train=True, download=True,
                       transform=trained_transform)
test = datasets.MNIST(root='./data', train=False, download=True,
                      transform=trained_transform)

# num_workers와 pin_memory를 추가하여 GPU 데이터 로딩 병목 현상 제거
train_loader = DataLoader(train, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


## 3. model 정의 및 장치 할당
model = SimpleMLP(input_dim= 28*28, hidden_dim=128, output_dim=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


## 4. 학습 루프
total_epochs = 10

for epoch in range(total_epochs):
  model.train()
  train_loss = 0

  for images, labels in train_loader:
    # 데이터를 GPU/CPU로 전송
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()

  avg_loss = train_loss / len(train_loader)

  # 출력 간격을 2 에포크로 단축 조정
  if (epoch+1) % 2 == 0:
    print(f"Epoch {epoch+1}/{total_epochs}, Train Loss: {avg_loss:.4f}")


## 5. 평가 함수 작성 & 평가 실행
def evaluate(model, data_loader):
  model.eval()
  correct = 0
  total = 0

  with torch.no_grad(): # 평가 단계에서 불필요한 기울기 계산 제거 → 메모리 효율 up
    for images, labels in data_loader:
      # 데이터를 GPU/CPU로 전송
      images, labels = images.to(device), labels.to(device)

      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0) # total: 전체 샘플 수 카운팅
      # correct: 예측과 실제 레이블이 같은 샘플 수만 카운팅
      correct += (predicted == labels).sum().item()

  accuracy = 100 * correct / total
  return accuracy

test_accuracy = evaluate(model, test_loader)
print(f"Test Accuracy: {test_accuracy:.2f}%")

Using device: cuda
Epoch 2/10, Train Loss: 1.2954
Epoch 4/10, Train Loss: 1.1990
Epoch 6/10, Train Loss: 1.1460
Epoch 8/10, Train Loss: 1.1272
Epoch 10/10, Train Loss: 1.1065
Test Accuracy: 63.93%


In [11]:
### Q4. Accuracy by learning rate, batch size, and epochs



In [12]:
### Q5. Using Pytorch's automatic differentiation

import torch

x = torch.tensor(2.0, requires_grad=True)
y = x*x + 3*x + 4 # y= x^2 + 3x + 4 이차함수

delta_y = x.backward()

print(f"x에 대한 자동 미분 기울기: {delta_y}")
print(f"x에 대한 실제 기울기: {x.grad}")

x에 대한 자동 미분 기울기: None
x에 대한 실제 기울기: 7.0


In [ ]:
### Q6. Observing the train curve with different learning rate

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# GPU 장치 설정 추가
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 단순 MLP 모델 정의
class SimpleMLP(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(SimpleMLP, self).__init__()
    self.input_dim = input_dim
    self.fc = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim)
    )

  def forward(self, x):
      x = x.view(-1, self.input_dim)
      return self.fc(x)


## 2. 훈련/테스트 데이터 호출 & 전처리/데이터로더 투입
trained_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.RandomResizedCrop(28),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
])

train = datasets.MNIST(root='./data', train=True, download=True,
                       transform=trained_transform)
train_loader = DataLoader(train, batch_size=64, shuffle=True)

criterion = nn.CrossEntropyLoss()
learning_rates = [0.001, 0.01, 0.1, 1.0]
colors = ['blue', 'green', 'red', 'yellow']

# 각 학습률별 20 에포크 동안의 손실 기록을 위한 딕셔너리
loss_history = {lr: [] for lr in learning_rates}


## 3. 학습률을 변화하며 독립적으로 학습 수행
for lr in learning_rates:
  print(f"Training with learning rate: {lr}...")
  # 각 학습률 마다 새로운 가중치를 갖는 모델 독립적으로 생성 후 GPU 전송
  model = SimpleMLP(input_dim=28*28, hidden_dim=128, output_dim=10).to(device)
  optimizer = optim.Adam(model.parameters(), lr=lr)

  for epoch in range(20):
    model.train()
    train_loss = 0

    for images, labels in train_loader:
      # 데이터를 GPU/CPU로 전송
      images, labels = images.to(device), labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      train_loss += loss.item()

    avg_loss = train_loss / len(train_loader)
    loss_history[lr].append(avg_loss)
    print(f"Epoch {epoch+1}/20, Loss: {avg_loss:.4f}")


## 4. 맷플롯립 격자 그래프 시각화 (1행 2열)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1) 좌측: 에포크당 훈련 손실 꺾은선 그래프
for lr, color in zip(learning_rates, colors):
  axes[0].plot(range(1, 21), loss_history[lr], label=f'LR = {lr}', color=color, marker='o')

axes[0].set_title("Training Curve Comparison")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_ylim(0, 2.5)
axes[0].set_yticks(np.arange(0, 2.51, 0.5))  # y축 0.5 간격
axes[0].set_xlim(0, 20)
axes[0].set_xticks(np.arange(0, 20.1, 2.5)) # x축 2.5 간격
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# 2) 우측: 최종 에포크(20th)의 손실값 비교 막대 그래프
final_losses = [loss_history[lr][-1] for lr in learning_rates]
axes[1].bar([str(lr) for lr in learning_rates], final_losses, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_title("Final Loss Comparison")
axes[1].set_xlabel("Learning Rate")
axes[1].set_ylabel("Final Loss")
axes[1].set_ylim(0, 2.5)
axes[1].set_yticks(np.arange(0, 2.51, 0.5))
axes[1].grid(True, axis='y', linestyle='--', alpha=0.6)

# 레이아웃 조절 및 출력
plt.tight_layout()
plt.show()


## 5. 분석 및 결론 출력
print("\n" + "="*50)
print(" [학습률별 분석 (Analysis by Learning Rate)]")
print("="*50)
print("  • LR = 0.001 (파랑): 안전하지만 느림 (점진적이고 안정적으로 손실이 감소함)")
print("  • LR = 0.01  (녹색): 균형잡힌 선택 (가장 빠르고 부드럽게 수렴 구간에 진입)")
print("  • LR = 0.1   (빨강): 빠르지만 불안정 (손실의 변동성이 크며 최적점에 정밀하게 도달하지 못함)")
print("  • LR = 1.0   (노랑): 너무 큰 보폭, 발산 (기울기가 폭발하거나 오버슈팅으로 학습이 이루어지지 않음)")
print("-"*50)

# 최적 학습률 도출
best_idx = np.argmin(final_losses)
best_lr = learning_rates[best_idx]

print(f"[실험 결론] 최종 에포크 기준 최적 학습률(Optimal LR): {best_lr}")
print("  - 학습률이 너무 작을 경우(e.g. 0.001): 가중치 업데이트가 너무 느려 수렴하는 데 많은 에포크가 소요된다.")
print("  - 학습률이 너무 클 경우(e.g. 1.0): 경사면을 따라 내려가지 못하고 발산(Divergence)하거나 최적점을 지나쳐 손실이 유지된다.")
print("="*50)

Using device: cuda
Training with learning rate: 0.001...
Epoch 1/20, Loss: 1.5500
